#### installing depedencies

In [1]:
%pip install requests beautifulsoup4 pandas lxml googlenewsdecoder

Note: you may need to restart the kernel to use updated packages.


#### import libraries

In [3]:
import time
from datetime import date, timedelta
from urllib.parse import quote_plus

import pandas as pd
import requests
from bs4 import BeautifulSoup
from googlenewsdecoder import gnewsdecoder

Keywords and variables

In [ ]:
psn = "Pasifik Satelit Nusantara"

SEARCH_QUERIES = [
    f'"{psn}"',
    '"PT Pasifik Satelit Nusantara"',
    '"Pasifik Satelit Nusantara" satellite',
    '"Pasifik Satelit Nusantara" internet',
    '"Nusantara Lima" PSN satelit',
    '"Nusantara Satu" PSN satelit',
    '"Nusantara Dua" PSN satelit',
    '"Nusantara Tiga" PSN satelit',
    'SATRIA PSN satellite',
    'PSN satelit Telkomsat',
    '"satria satu"',
    '"satria 1"',
    '"nusantara tiga"',
    '"satelit satria satu"',
    '"satelit nusantara tiga"',
]

MAX_ARTICLES = 1000          
DAYS_BACK = 365 * 10   # days back to search for articles     
CHUNK_DAYS = 365             
GET_FULL_TEXT = True         
MAX_FULL_TEXT = MAX_ARTICLES 
PAUSE_SECONDS = 1            
SEARCH_PAUSE_SECONDS = 0.5   
CHECKPOINT_EVERY = 25        
OUTPUT_CSV = "psn_news_scraped.csv"    
OUTPUT_JSON = "psn_news_scraped.json" 

#### getting news / article data helper function 

In [6]:

BROWSER_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "id-ID,id;q=0.9,en;q=0.8",   # indonesian and english
}


def getNewsData(url):
    """Fetch and parse one Google News RSS search with BeautifulSoup."""
    response = requests.get(url, headers=BROWSER_HEADERS, timeout=20)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "xml")
    news_results = []   

    
    for item in soup.find_all("item"):
        link = item.find("link")
        title = item.find("title")
        description = item.find("description")
        pub_date = item.find("pubDate")
        source = item.find("source")

        
        if not link or not link.get_text(strip=True):
            continue

        
        snippet_html = description.get_text(" ", strip=True) if description else ""
        snippet = BeautifulSoup(snippet_html, "html.parser").get_text(" ", strip=True)

        news_results.append({
            "link": link.get_text(strip=True),     
            "title": title.get_text(" ", strip=True) if title else "",
            "snippet": snippet,                      
            "date": pub_date.get_text(strip=True) if pub_date else "",
            "source": source.get_text(" ", strip=True) if source else "",   
            "article_url": "",    
            "article_text": "",   
        })

    return news_results  

#### Getting article text helper function

In [ ]:
def getArticleText(url):
    """Resolve a Google News link and extract publisher paragraphs with BeautifulSoup."""

    decoded = gnewsdecoder(url, interval=1)
    article_url = decoded.get("decoded_url")                      
    decode_succeeded = decoded.get("success") or decoded.get("status")  
    
    if not decode_succeeded or not article_url:
        raise ValueError(f"Could not resolve Google News link: {decoded}")

    
    response = requests.get(
        article_url,
        headers=BROWSER_HEADERS,    
        timeout=20,                 
        allow_redirects=True,       
    )
    response.raise_for_status()     

    
    soup = BeautifulSoup(response.content, "html.parser")

    content = soup.find("article") or soup.find("main") or soup.body or soup

    paragraphs = [
        paragraph.get_text(" ", strip=True)
        for paragraph in content.find_all("p")
        if paragraph.get_text(strip=True)
    ]

    return response.url, "\n".join(paragraphs)

#### list of search urls

In [ ]:

editions = [
    "hl=id&gl=ID&ceid=ID:id",   # Indonesian news, Indonesia
    "hl=en&gl=US&ceid=US:en",   # English news, USA
]


end_date = date.today() + timedelta(days=1)             
start_date = end_date - timedelta(days=DAYS_BACK)       


date_chunks = []                  
chunk_start = start_date
while chunk_start < end_date:     
    
    chunk_end = min(chunk_start + timedelta(days=CHUNK_DAYS), end_date)
    date_chunks.append((chunk_start, chunk_end))
    chunk_start = chunk_end       

date_chunks.reverse()   


links_psn = []   
for chunk_start, chunk_end in date_chunks:
    for query in SEARCH_QUERIES:
        
        dated_query = (
            f"{query} after:{chunk_start.isoformat()} "
            f"before:{chunk_end.isoformat()}"
        )
        for edition in editions:
            links_psn.append(
                "https://news.google.com/rss/search?q="
                + quote_plus(dated_query)    
                + "&"
                + edition
            )

print(f"Prepared {len(links_psn)} dated Google News searches.")   

Prepared 300 dated Google News searches.


#### Scrapping 

In [ ]:
news_by_title = {}   


for search_number, link in enumerate(links_psn, start=1):
    try:
        search_results = getNewsData(link)       
    except requests.RequestException as error:   
        print(f"Search {search_number}/{len(links_psn)} failed: {error}")
        continue                                 

    for article in search_results:
        title_key = " ".join(article["title"].casefold().split())
        if title_key and title_key not in news_by_title:
            news_by_title[title_key] = article
        if len(news_by_title) >= MAX_ARTICLES:
            break

    if search_number % 10 == 0:
        print(
            f"Searched {search_number}/{len(links_psn)} feeds; "
            f"collected {len(news_by_title)} unique headlines."
        )
    if len(news_by_title) >= MAX_ARTICLES:   
        break
    time.sleep(SEARCH_PAUSE_SECONDS)         

news_results = list(news_by_title.values())


def save_results():
    columns = ["link", "title", "snippet", "date", "source", "article_url", "article_text"]
    df_psn = pd.DataFrame(news_results, columns=columns)   
    df_psn.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")   
    df_psn.to_json(OUTPUT_JSON, orient="records", force_ascii=False, indent=2)
    return df_psn


df_psn = save_results()


if GET_FULL_TEXT:   
    articles_to_fetch = news_results[:MAX_FULL_TEXT]   
    for index, article in enumerate(articles_to_fetch, start=1):
        try:
            article["article_url"], article["article_text"] = getArticleText(article["link"])
            if not article["article_text"].strip():
                print(f"No paragraph text found: {article['title']}")
        except (requests.RequestException, ValueError) as error:
            print(f"Could not fetch article ({article['title']}): {error}")

        
        if index % CHECKPOINT_EVERY == 0 or index == len(articles_to_fetch):
            df_psn = save_results()
            print(f"Fetched full text for {index}/{len(articles_to_fetch)} articles.")
        time.sleep(PAUSE_SECONDS)   


df_psn = save_results()
print(f"Collected {len(df_psn)} news items. CSV saved to {OUTPUT_CSV}.")
if len(df_psn) < 200:
    print("Fewer than 200 articles were found; Google News availability varies by query and date.")
df_psn.head()   

Searched 10/300 feeds; collected 58 unique headlines.
Searched 20/300 feeds; collected 74 unique headlines.
Searched 30/300 feeds; collected 147 unique headlines.
Searched 40/300 feeds; collected 235 unique headlines.
Searched 50/300 feeds; collected 249 unique headlines.
Searched 60/300 feeds; collected 289 unique headlines.
Searched 70/300 feeds; collected 313 unique headlines.
Searched 80/300 feeds; collected 332 unique headlines.
Searched 90/300 feeds; collected 436 unique headlines.
Searched 100/300 feeds; collected 490 unique headlines.
Searched 110/300 feeds; collected 504 unique headlines.
Searched 120/300 feeds; collected 571 unique headlines.
Searched 130/300 feeds; collected 588 unique headlines.
Searched 140/300 feeds; collected 592 unique headlines.
Searched 150/300 feeds; collected 645 unique headlines.
Searched 160/300 feeds; collected 670 unique headlines.
Searched 170/300 feeds; collected 679 unique headlines.
Searched 180/300 feeds; collected 713 unique headlines.
Sea

,link,title,snippet,date,source,article_url,article_text
0,https://news.google.com/rss/articles/CBMia0FVX...,Berita Kini - Menkomdigi Meutya Hafid: Satelit...,Berita Kini - Menkomdigi Meutya Hafid: Satelit...,"Tue, 12 May 2026 07:00:00 GMT",Kementerian Komunikasi dan Digital,https://portal.komdigi.go.id/kanal-publik/beri...,Menteri Komunikasi dan Digital Meutya Hafid me...
1,https://news.google.com/rss/articles/CBMikwFBV...,KSP Dukung Pemerataan Internet di Daerah 3T Me...,KSP Dukung Pemerataan Internet di Daerah 3T Me...,"Tue, 12 May 2026 07:00:00 GMT",Kantor Staf Presiden,https://www.ksp.go.id/ksp-dukung-pemerataan-in...,"Kepala Staf Kepresidenan, Jenderal TNI (Purn.)..."
2,https://news.google.com/rss/articles/CBMiswFBV...,"Perebutan 'Kue' Internet Satelit RI Memanas, P...","Perebutan 'Kue' Internet Satelit RI Memanas, P...","Wed, 13 May 2026 07:00:00 GMT",detikInet,https://inet.detik.com/telecommunication/d-848...,Agus Tri Haryanto - detikInet\nDirektur Utama ...
3,https://news.google.com/rss/articles/CBMihwFBV...,Jejak Satelit Nusantara Lima Menuju Orbit - Ko...,Jejak Satelit Nusantara Lima Menuju Orbit Komp...,"Sun, 01 Feb 2026 08:00:00 GMT",Kompas.id,https://www.kompas.id/artikel/jejak-satelit-nu...,Satelit Nusantara Lima tengah menjalani proses...
4,https://news.google.com/rss/articles/CBMi0gFBV...,Satelit Nusantara Lima Menjadi Satelit Terbesa...,Satelit Nusantara Lima Menjadi Satelit Terbesa...,"Thu, 14 May 2026 07:00:00 GMT",SWA.co.id,https://swa.co.id/read/472559/satelit-nusantar...,"Dilarang keras mengambil konten (tulisan, foto..."


#### Checking the scraped news / articles

In [10]:
df_psn.sample(10)

,link,title,snippet,date,source,article_url,article_text
437,https://news.google.com/rss/articles/CBMiogFBV...,PSN: Satelit Satria akan ciptakan pemerataan a...,PSN: Satelit Satria akan ciptakan pemerataan a...,"Thu, 22 Jun 2023 07:00:00 GMT",ANTARA News,https://www.antaranews.com/berita/3601977/psn-...,Jakarta (ANTARA) - PT Pasifik Satelit Nusantar...
622,https://news.google.com/rss/articles/CBMilgFBV...,President Jokowi: New Capital Part of Indonesi...,President Jokowi: New Capital Part of Indonesi...,"Tue, 18 Jan 2022 07:49:51 GMT",Sekretariat Kabinet Republik Indonesia,https://setkab.go.id/en/president-jokowi-new-c...,
834,https://news.google.com/rss/articles/CBMirAFBV...,"Satelit Nusantara Satu Sukses Meluncur, Kapan ...","Satelit Nusantara Satu Sukses Meluncur, Kapan ...","Sat, 23 Feb 2019 08:00:00 GMT",HiTekno.com,https://www.hitekno.com/sains/2019/02/23/08150...,"Satelit Nusantara Satu Sukses Meluncur, Kapan ..."
362,https://news.google.com/rss/articles/CBMiugFBV...,Mengenal Fungsi Penting Stasiun Kendali Bumi S...,Mengenal Fungsi Penting Stasiun Kendali Bumi S...,"Tue, 12 Dec 2023 08:00:00 GMT",detikInet,https://inet.detik.com/telecommunication/d-708...,Anggoro Suryo - detikInet\nSatelit Republik In...
549,https://news.google.com/rss/articles/CBMiyAFBV...,"Satelit SATRIA-1 Berhasil Ngorbit, Bersiap Kas...","Satelit SATRIA-1 Berhasil Ngorbit, Bersiap Kas...","Tue, 31 Oct 2023 07:00:00 GMT",CNN Indonesia,https://www.cnnindonesia.com/teknologi/2023103...,Satelit Republik Indonesia atau SATRIA-1 berha...
725,https://news.google.com/rss/articles/CBMinwFBV...,Satelit Nusantara 2 yang Gagal Orbit Sudah Dia...,Satelit Nusantara 2 yang Gagal Orbit Sudah Dia...,"Sat, 11 Apr 2020 07:00:00 GMT",Tempo.co,,
774,https://news.google.com/rss/articles/CBMikAJBV...,First discovery of subdichthadiigyne in Yunodo...,First discovery of subdichthadiigyne in Yunodo...,"Wed, 01 Jan 2020 08:00:00 GMT",BioOne Complete,https://bioone.org/journals/revue-suisse-de-zo...,
405,https://news.google.com/rss/articles/CBMihgFBV...,Upaya Mempercepat Aksesibilitas Internet di Wi...,Upaya Mempercepat Aksesibilitas Internet di Wi...,"Sun, 10 Dec 2023 08:00:00 GMT",Tempo.co,,
249,https://news.google.com/rss/articles/CBMiqAFBV...,"Komdigi Usul Tambahan Rp 12,6 T untuk Perawata...","Komdigi Usul Tambahan Rp 12,6 T untuk Perawata...","Mon, 07 Jul 2025 07:00:00 GMT",detikNews,https://news.detik.com/berita/d-7999687/komdig...,Kementerian Komunikasi dan Digital ( Komdigi )...
594,https://news.google.com/rss/articles/CBMivAFBV...,Restoran di Grand Mall Bekasi Dilanda Kebakara...,Restoran di Grand Mall Bekasi Dilanda Kebakara...,"Tue, 08 Mar 2022 08:00:00 GMT",Kompas.com,https://megapolitan.kompas.com/read/2022/03/08...,Restoran di Grand Mall Bekasi Dilanda Kebakara...
